# ChainLens Phase 3 — Feature EDA

Read-only exploration of `address_features`. Split/label joins are for **visualisation only** and are never written back. Run after:

```bash
CHAINLENS_SAMPLE_MODE=1 make features
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sqlalchemy import text

from chainlens.db import get_pg_engine
from chainlens.features import SCALER_SPECS

pd.set_option("display.max_columns", 50)
engine = get_pg_engine()

features = pd.read_sql(text("SELECT * FROM address_features"), engine)
splits = pd.read_sql(text("SELECT address, split FROM label_splits"), engine)
df = features.merge(splits, on="address", how="left")
df["split"] = df["split"].fillna("unlabeled")
print(f"address_features: {len(features):,} rows / {features['address'].nunique():,} addresses")
print(df["split"].value_counts().to_string())


## Raw and scaled distributions

Each histogram is split-coloured. Unlabelled fixture addresses are shown as context but are never used for scaling or evaluation.

In [ ]:
raw_cols = [
    "tx_count_out", "tx_count_in", "unique_counterparties",
    "total_value_eth", "fanout_ratio",
]
fig, axes = plt.subplots(len(raw_cols), 2, figsize=(12, 3 * len(raw_cols)))
for row, col in enumerate(raw_cols):
    for split in ["train", "test", "unlabeled"]:
        values = df.loc[df["split"] == split, col].astype(float).dropna()
        if values.empty:
            continue
        axes[row, 0].hist(values, bins=30, alpha=0.45, label=split)
        axes[row, 1].hist(values, bins=30, alpha=0.45, label=split)
    axes[row, 0].set_title(f"{col} (raw)")
    axes[row, 1].set_title(f"{col} (log1p view)")
    axes[row, 1].set_xscale("symlog")
    axes[row, 0].legend()
plt.tight_layout()
plt.show()


## Skewness before and after log1p

Only features whose scaler transform is `log1p_std` are shown.

In [ ]:
rows = []
for feature, transform in SCALER_SPECS.items():
    if transform != "log1p_std":
        continue
    raw = df[feature].astype(float).dropna()
    rows.append({
        "feature": feature,
        "skew_raw": raw.skew(),
        "skew_log1p": np.log1p(raw).skew(),
    })
skew_table = pd.DataFrame(rows).set_index("feature").round(3)
skew_table


## Expected evaluable positives

This is the number Phase 6 can actually evaluate on: **test-split addresses present in `address_features` with at least one transaction**. It is deliberately printed, not written into `address_features`.

In [ ]:
expected = pd.read_sql(text('''
    SELECT count(DISTINCT af.address) AS n
    FROM address_features af
    JOIN label_splits ls ON ls.address = af.address
    WHERE ls.split = 'test' AND af.tx_count_total >= 1
'''), engine).iloc[0]["n"]
print("=" * 60)
print(f"EXPECTED EVALUABLE POSITIVES: {expected:,}")
print("=" * 60)


## Train-side class balance by source

The labels table is joined for display only. No `flagged` column is created anywhere.

In [ ]:
labels = pd.read_sql(text("SELECT address, source, label FROM labels"), engine)
train = df.loc[df["split"] == "train", ["address"]].merge(labels, on="address", how="inner")
balance = (
    train.groupby(["source", "label"], as_index=False)
    .size()
    .rename(columns={"size": "addresses"})
    .sort_values(["source", "label"])
)
balance


## Scaler provenance and dropped features

In [ ]:
scaler = pd.read_sql(
    text("SELECT feature_name, transform, mean, std, fitted_on_split, row_count, scaler_version "
         "FROM feature_scaler ORDER BY feature_name"),
    engine,
)
dropped = pd.read_sql(
    text("SELECT feature_name, reason FROM feature_scaler_dropped ORDER BY feature_name"),
    engine,
)
print(f"scaler_version: {df['scaler_version'].dropna().unique().tolist()}")
display(scaler)
display(dropped)


## Notes

- Scaler rows are fitted on the train split only; every `_scaled` column is applied to all addresses.
- Dropped zero-variance features have `NULL` scaled values and are excluded from `feature_scaler`.
- `net_flow_eth` is intentionally raw; it is not part of the locked scaled-column list.